In [37]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
from shapely.geometry import Point
os.getcwd()

'c:\\Users\\jmc753\\Work\\occ-coseismic'

In [38]:
def buffer_coastline(coastline, buffer, directory="./sites/poly_buffers", tabs=1):
    os.makedirs(directory, exist_ok=True)
    tab = '\t'
    print(f"{tab * tabs}Buffering coastline...")
    buffer_file = f"{directory}/coastline_{buffer}m.geojson"

    if os.path.exists(buffer_file):
        print(f"{tab * (tabs + 1)}Reading {buffer} m coast buffer...")
        buffer_poly = gpd.read_file(buffer_file).geometry
    else:
        print(f"{tab * (tabs + 1)}Creating {buffer} m coast buffer")
        buffer_poly = coastline.buffer(buffer)
        buffer_poly.to_file(buffer_file, driver='GeoJSON')
    
    return buffer_poly

In [ ]:
coastline_file = ".\\data\\coastline\\nz-coastlines-and-islands-polylines-topo-150k.gpkg"
coastpoly_file = ".\\data\\coastline\\nz-coastlines-and-islands-polygons-topo-150k.gpkg"
fault_file = 'C:\\Users\\jmc753\\Work\\NZ_CFM_v1_0\\Shapefiles\\NZ_CFM_v1_0.shp'
out_file = '.\\sites\\EastCoastNI_3km.csv'

backbone_res_km = 0
coast_filters = [[1000, 500]]   # [Site Resolution, Coast Buffer]. Either in m, or if > 1km, must be 1km increments
coast_filters = [[3000, 1500]]
fault_trace_filters = [[1000, 500, 1]]   # [Site Resolution, Fault Buffer, > SlipRate]
fault_trace_filters = [[]]
fault_poly_buffers = [['CFM', 15000, 6000, 3000, 0.49, 3000], ['NW Cardrona North', 15000, 6000, 3000, 0.49, 3000], ['Inglewood', 15000, 6000, 3000, 0.49, 3000],
                      ['py', 15000, 6000, 3000, 0.35, 3000], ['sz', 15000, 6000, 3000, 0.35, 3000]]  #  [Fault types, hanging buffer, footbuffer, edgebuffer, min slip rate, site resolution]
fault_poly_buffers = [['sz', 45000, 6000, 3000, 0.35, 3000]]

premade_points = ['.\\sites\\v0-2-faults_points.geojson', '.\\sites\\v0-2-faults-CFM_points.geojson']
premade_points = []

final_coast_buffer = 0   # Final coast clip

min_area = 1e6  # sq km

if backbone_res_km > 0:
    backbone = gpd.read_file(f".\\sites\\national_{backbone_res_km}km_grid.geojson")
else:
    backbone = gpd.GeoDataFrame(columns=['geometry'], geometry='geometry', crs='EPSG:2193')

coastline = gpd.read_file(coastline_file)
coastline = coastline[coastline['Area_SquareMeter'] > min_area]

coast_poly = gpd.read_file(coastpoly_file)
coast_poly = coast_poly[coast_poly['Area_SquareMeter'] > min_area]

faults = gpd.read_file(fault_file)
faults = gpd.sjoin(faults, coast_poly, how="inner", predicate="intersects")

os.makedirs('.\\sites\\clipped_sites', exist_ok=True)

coast_filters = coast_filters if len(coast_filters) > 0 else [[0, 0]]
fault_trace_filters = fault_trace_filters if len(fault_trace_filters) > 0 else [[]]
fault_poly_buffers = fault_poly_buffers if len(fault_poly_buffers) > 0 else [[]]

In [40]:
points = gpd.GeoDataFrame(columns=['geometry'], geometry='geometry', crs='EPSG:2193')
n_premade = np.zeros(len(premade_points), dtype=int)
if len(premade_points) > 0:
    print(f"Adding premade points...")
    for i, point_file in enumerate(premade_points):
        points_file = point_file.replace('.csv', '.geojson') if point_file.endswith('.csv') else point_file
        if not os.path.exists(points_file):
            print(f"\t{point_file} does not exist, skipping...")
            continue
        print(f"\tReading from {point_file}...")
        new_points = gpd.read_file(point_file)
        n_premade[i] = new_points.shape[0]
        print(f"\t\tAdding {n_premade[i]} points...")
        points = pd.concat([points, new_points])
        print(f"\t\t{points.shape[0]} total points currently...")


print(f"\nAdding points from national {backbone_res_km} km backbone resolution...")
print(f"\tAdding {backbone.shape[0]} points...")
points = pd.concat([points, backbone[['geometry']]])
print(f"\t{points.shape[0]} total points currently...")



Adding points from national 0 km backbone resolution...
	Adding 0 points...
	0 total points currently...


In [41]:
print(f"Adding coastal points from {coastline.shape[0]} islands above {min_area * 1e-6:.0f} sqkm")
coast_arr = np.array(coast_filters)
if coast_arr.shape[1] > 0:
    coast_arr = coast_arr[np.lexsort((coast_arr[:, 1], coast_arr[:, 0]))]
n_coast_points = np.zeros(coast_arr.shape[0], dtype=np.int32)

for ix, (res, buffer) in enumerate(coast_arr[1:], 1):
    res_units = 'm' if res < 1000 else 'km'
    res = int(res) if res < 1000 else int(res / 1000)
    buffer_units = 'm' if buffer < 1000 else 'km'
    print(f"\t{res} {res_units} spacing to {buffer if buffer < 1000 else int(buffer / 1000)} {buffer_units} from coast")
    buffed_file = f'.\\sites\\clipped_sites\\coast_{min_area * 1e-6:.0f}_sqkm_{buffer if buffer < 1000 else int(buffer / 1000)}_{buffer_units}_buff_{res}_{res_units}_res.geojson'
    if os.path.exists(buffed_file):
        print(f"\t\tReading pre-prepared {buffed_file}...")
        new_points = gpd.read_file(buffed_file)
    else:
        print(f"\t\tReading national_{res}{res_units}_grid.geojson...")
        grid = gpd.read_file(f".\\sites\\national_{res}{res_units}_grid.geojson")
        buffer_poly = buffer_coastline(coastline, buffer, tabs=2)
        print("\t\tFinding points in buffer...")
        intersect = grid.intersects(buffer_poly.unary_union)
        new_points = grid[intersect].reset_index()[['geometry']]
        new_points.to_file(buffed_file)

    n_coast_points[ix] = new_points.shape[0]
    print(f"\t\tAdding {n_coast_points[ix]} points...")
    points = pd.concat([points, new_points])
    print(f"\t\t{points.shape[0]} total points currently...")

Adding coastal points from 90 islands above 1 sqkm


In [42]:
print(f"Adding fault_adjacent points....")
n_fault_points = np.zeros(len(fault_trace_filters), dtype=np.int32)

if len(fault_trace_filters[0]) > 0:
    for ix, (res, buffer, min_slip_rate) in enumerate(np.array(fault_trace_filters)):
        res_units = 'm' if res < 1000 else 'km'
        res = int(res) if res < 1000 else int(res / 1000)
        buffer_units = 'm' if buffer < 1000 else 'km'
        print(f"\t{res} {res_units} spacing to {buffer if buffer < 1000 else int(buffer / 1000)} {buffer_units} from {min_slip_rate} mm/yr faults")
        buffed_file = f'.\\sites\\clipped_sites\\faults_{str(min_slip_rate).replace(".", "-")}_mm_{buffer if buffer < 1000 else int(buffer / 1000)}_{buffer_units}_buff_{res}_{res_units}_res.geojson'
        if os.path.exists(buffed_file):
            print(f"\t\tReading pre-prepared {buffed_file}...")
            new_points = gpd.read_file(buffed_file)
        else:
            print(f"\t\tReading national_{res}{res_units}_grid.geojson...")
            grid = gpd.read_file(f".\\sites\\national_{res}{res_units}_grid.geojson")
            print("\t\tBuffering faults...")
            buffer_poly = faults[faults.SR_pref > min_slip_rate].buffer(buffer)
            print("\t\tFinding points in buffer...")
            intersect = grid.intersects(buffer_poly.unary_union)
            new_points = grid[intersect].reset_index()[['geometry']]
            new_points.to_file(buffed_file)

        n_fault_points[ix] = new_points.shape[0]
        print(f"\t\tAdding {n_fault_points[ix]} points...")
        points = pd.concat([points, new_points])
        print(f"\t\t{points.shape[0]} total points currently...")

Adding fault_adjacent points....


In [43]:
print(f"Adding Fault Polygon Buffers")
n_poly_points = np.zeros(len(fault_poly_buffers), dtype=np.int32)
if len(fault_poly_buffers[0]) > 0:
    for ix, (fault_type, hangingbuff, footbuff, edgebuff, min_slip, res) in enumerate(fault_poly_buffers):
        fault_type = fault_type.replace(' ', '-')
        res_units = 'm' if res < 1000 else 'km'
        res = int(res) if res < 1000 else int(res / 1000)
        if fault_type == 'CFM':
            print(f"\t{res} {res_units} in > {str(min_slip).replace('.', '-')} mm/yr {fault_type} for {hangingbuff / 1000:.0f}/{footbuff / 1000:.0f}/{edgebuff / 1000:.0f}km buffers")
            poly_buffer_file = f"{fault_type}_hang-{hangingbuff / 1000:.0f}km_foot-{footbuff / 1000:.0f}km_edge-{edgebuff / 1000:.0f}km_gt{str(min_slip).replace('.', '-')}mmyr.geojson"
        else:
            print(f"\t{res} {res_units} in {fault_type} for {hangingbuff / 1000:.0f}/{footbuff / 1000:.0f}/{edgebuff / 1000:.0f}km buffers")
            poly_buffer_file = f"{fault_type}_hang-{hangingbuff / 1000:.0f}km_foot-{footbuff / 1000:.0f}km_edge-{edgebuff / 1000:.0f}km.geojson"
        if os.path.exists(f"./sites/poly_buffers/{poly_buffer_file}"):
            buffer_poly = gpd.read_file(f"./sites/poly_buffers/{poly_buffer_file}")
        else:
            print(f"\t./sites/poly_buffers/{poly_buffer_file} not found. Run generate_polygon_buffer.ipynb")
            continue

        print(f"\t\tReading national_{res}{res_units}_grid.geojson...")
        grid = gpd.read_file(f".\\sites\\national_{res}{res_units}_grid.geojson")
        print("\t\tFinding points in buffer...")
        intersect = grid.intersects(buffer_poly.unary_union)
        new_points = grid[intersect].reset_index()[['geometry']]
        n_poly_points[ix] = new_points.shape[0]
        print(f"\t\tAdding {n_poly_points[ix]} points...")
        points = pd.concat([points, new_points])
        print(f"\t\t{points.shape[0]} total points currently...")


Adding Fault Polygon Buffers
	3 km in sz for 45/6/3km buffers
		Reading national_3km_grid.geojson...
		Finding points in buffer...
		Adding 9239 points...
		9239 total points currently...


In [44]:
print('Cropping to Island Poly')
points = gpd.sjoin(points, coast_poly, predicate="within")
print(f'\t{points.shape[0]} points...')

if coast_arr.shape[1] > 0 and coast_arr[0, 1] > 0:
    print(f"Adding highest res coast buffer (allows some offshore)")
    res, buffer = coast_arr[0]
    res_units = 'm' if res < 1000 else 'km'
    res = res if res < 1000 else int(res / 1000)
    buffer_units = 'm' if buffer < 1000 else 'km'
    print(f"\t{res} {res_units} spacing to {buffer if buffer < 1000 else int(buffer / 1000)} {buffer_units} from coast")
    buffed_file = f'.\\sites\\clipped_sites\\coast_{min_area * 1e-6:.0f}_sqkm_{buffer if buffer < 1000 else int(buffer / 1000)}_{buffer_units}_buff_{res}_{res_units}_res.geojson'
    if os.path.exists(buffed_file):
        print(f"\t\tReading pre-prepared {buffed_file}...")
        new_points = gpd.read_file(buffed_file)
    else:
        print(f"\t\tReading national_{res}{res_units}_grid.geojson...")
        grid = gpd.read_file(f".\\sites\\national_{res}{res_units}_grid.geojson")
        buffer_poly = buffer_coastline(coastline, buffer, tabs=2)
        print("\t\tFinding points in buffer...")
        intersect = grid.intersects(buffer_poly.unary_union)
        new_points = grid[intersect].reset_index()[['geometry']]
        new_points.to_file(buffed_file)

    n_coast_points[0] = new_points.shape[0]
    print(f"\t\tAdding {n_coast_points[0]} points...")
    points = pd.concat([points, new_points])
    print(f"\t\t{points.shape[0]} total points currently...")

Cropping to Island Poly
	8734 points...
Adding highest res coast buffer (allows some offshore)
	3 km spacing to 1 km from coast
		Reading pre-prepared .\sites\clipped_sites\coast_1_sqkm_1_km_buff_3_km_res.geojson...
		Adding 3533 points...
		12267 total points currently...


In [45]:
if final_coast_buffer > 0:
    previous_points = points.shape[0]
    print(f"Running {final_coast_buffer / 1000:.1f}km Final Coast Crop")
    buffer_poly = buffer_coastline(coastline, final_coast_buffer, tabs=1)
    print("\tFinding points in buffer...")
    intersect = points.intersects(buffer_poly.unary_union)
    points = points[intersect].reset_index()[['geometry']]
    print(f"\t\t{points.shape[0]} total points currently...")


In [46]:
print('Formatting....')
points = points.reset_index()
points['siteId'] = ''
points['Lon'] = np.round(points.geometry.x, 1)
points['Lat'] = np.round(points.geometry.y, 1)
points['Height'] = 0

points['siteId'] = [f"{round(points.loc[ix, 'Lon'])}_{round(points.loc[ix, 'Lat'])}" for ix in points.index.values]  # Set siteId to be based on NZTM location

points = points[['siteId', 'Lon', 'Lat', 'Height', 'geometry']]

print(f'\t{points.shape[0]} points')
print('Remove duplicate points...')
points = points.drop_duplicates(ignore_index=True)
print(f'\t{points.shape[0]} points')

Formatting....
	12267 points
Remove duplicate points...
	11889 points


In [47]:
print(f'Writing outputs...')
points[['siteId', 'Lon', 'Lat', 'Height']].to_csv(out_file, index=False)
print(f"\tOutput file saved as: {out_file}")
points.to_file(out_file.replace('.csv', '.geojson'), driver='GeoJSON')
print(f"\tOutput GeoJSON file saved as: {out_file.replace('.csv', '.geojson')}")

print("Splitting outputs into Hikurangi and Puysegur sections...")
wellington = Point([1749150, 5428092]) # Wellington coordinates in NZTM
te_anau = Point([1186710, 4957633])  # Te Anau coordinates in NZTM
distance = 350  # Distance South of Wellington in km to include for hikurangi

# For Hikurangi, find all centroids north of 350km south of Wellington
northern_section = points[(points.geometry.y > wellington.y) | (points.distance(wellington) < distance * 1e3)]
northern_section.to_file(out_file.replace('.csv', 'N.geojson'), driver='GeoJSON')
print(f"\t{northern_section.shape[0]} Northern sites")
print(f"\tWritten {out_file.replace('.csv', 'N.geojson')}")
northern_section[['siteId', 'Lon', 'Lat', 'Height']].to_csv(out_file.replace('.csv', 'N.csv'), index=False)
print(f"\tWritten {out_file.replace('.csv', 'N.csv')}")

# For Puysegur, find all centroids within 350km of Te Anau
southern_section = points[(points.distance(te_anau) < distance * 1e3)]
southern_section.to_file(out_file.replace('.csv', 'S.geojson'), driver='GeoJSON')
print(f"\t{southern_section.shape[0]} Southern sites")
print(f"\tWritten {out_file.replace('.csv', 'S.geojson')}")
southern_section[['siteId', 'Lon', 'Lat', 'Height']].to_csv(out_file.replace('.csv', 'S.csv'), index=False)
print(f"\tWritten {out_file.replace('.csv', 'S.csv')}")


Writing outputs...
	Output file saved as: .\sites\EastCoastNI_3km.csv
	Output GeoJSON file saved as: .\sites\EastCoastNI_3km.geojson
Splitting outputs into Hikurangi and Puysegur sections...
	10787 Northern sites
	Written .\sites\EastCoastNI_3kmN.geojson
	Written .\sites\EastCoastNI_3kmN.csv
	870 Southern sites
	Written .\sites\EastCoastNI_3kmS.geojson
	Written .\sites\EastCoastNI_3kmS.csv


In [48]:
print('Writing MetaData')
with open(out_file.replace('.csv', '_meta.txt'), 'w') as f:
    f.write(f"Name: {os.path.basename(out_file)}\n\n")
    f.write(f"National Resolution: {backbone_res_km} km ({backbone.shape[0]})\n")
    f.write(f"\nCoastal Strips:\n")
    if coast_arr.shape[1] > 0:
        for ix, (res, buffer) in enumerate(coast_arr):
            res_units = 'm' if res < 1000 else 'km'
            res = res if res < 1000 else int(res / 1000)
            buffer_units = 'm' if buffer < 1000 else 'km'
            offshore = ", inc. offshore" if ix == 0 else ""
            f.write(f"\t{ix}: {res} {res_units} spacing in {buffer if buffer < 1000 else int(buffer / 1000)} {buffer_units} coast buffer ({n_coast_points[ix]}{offshore})\n")
    f.write(f"\nFault Traces:\n")
    if len(fault_trace_filters[0]) > 0:
        for ix, (res, buffer, min_slip_rate) in enumerate(fault_trace_filters):
            res_units = 'm' if res < 1000 else 'km'
            res = res if res < 1000 else int(res / 1000)
            buffer_units = 'm' if buffer < 1000 else 'km'
            f.write(f"\t{ix}: {res} {res_units} spacing for {buffer if buffer < 1000 else int(buffer / 1000)} {buffer_units} around >{min_slip_rate} mm/yr faults ({n_fault_points[ix]})\n")
    f.write(f"\nFault Polygons:\n")
    if len(fault_poly_buffers[0]) > 0:
        for ix, (fault_type, hangingbuff, footbuff, edgebuff, min_slip, res) in enumerate(fault_poly_buffers):
            res_units = 'm' if res < 1000 else 'km'
            res = res if res < 1000 else int(res / 1000)
            if fault_type == 'CFM':
                f.write(f"\t{ix}: {res} {res_units} in > {str(min_slip).replace('.', '-')} mm/yr {fault_type} for {hangingbuff / 1000:.0f}/{footbuff / 1000:.0f}/{edgebuff / 1000:.0f}km buffers ({n_poly_points[ix]})\n")
            else:
                f.write(f"\t{ix}: {res} {res_units} in {fault_type} for {hangingbuff / 1000:.0f}/{footbuff / 1000:.0f}/{edgebuff / 1000:.0f}km buffers ({n_poly_points[ix]})\n")

    if len(premade_points) > 0:
        f.write(f"\nPremade Points:\n")
        for ix, point_file in enumerate(premade_points):
            f.write(f"\t{ix}: {point_file} ({n_premade[ix]})\n")

    if final_coast_buffer > 0:
        f.write(f"\nFinal Coast Crop: {final_coast_buffer} m - {previous_points} sites before cropping\n")
    f.write(f"\nTotal sites: {points.shape[0]}\n")
    f.write(f"Northern sites: {northern_section.shape[0]}\n")
    f.write(f"Southern sites: {southern_section.shape[0]}\n")
    

Writing MetaData
